In [ ]:
import numpy as np
import pandas as pd

In [ ]:
#carregando o dataframe

df = pd.read_csv('dados_conab/relatorio_preco_medio_mensal_2023.csv')


In [ ]:
#pedindo 20 linhas aleatórias de exemplo dos dados

df.sample(20)

In [ ]:
#pedindo as primeiras 20 linhas

df.head(20)

In [ ]:
#pedindo as últimas 20 linhas

df.tail(20)

In [ ]:
#eliminar as últimas 5 linhas, que não tem dados

df = df.iloc[:-5]
df.tail(20)

In [ ]:
#preencher linhas em branco com o mesmo valor da última linha preenchida acima nas colunas Produto/Unidade e Nível de Comercialização

df['Produto/Unidade'] = df['Produto/Unidade'].ffill()
df['Nível de Comercialização'] = df['Nível de Comercialização'].ffill()

df['Produto/Unidade'].isnull().sum(), df['Nível de Comercialização'].isnull().sum()

In [ ]:
print(df.head)


In [ ]:
df.head(20)

In [ ]:
#separa produto e unidade de medida e cria duas colunas. Pois, tem o mesmo produto em KG, toneladam, unidade, dúzia, litro... misturados na coluna Produto/Unidade
extracted = df['Produto/Unidade'].str.extract(r'^(.*?)\s*\(([^)]+)\)\s*$')
df['produto_nome'] = extracted[0].str.strip()
df['unidade'] = extracted[1].str.strip()
df.head(20)

In [ ]:
#converte os preços de texto para número
meses = ['01/2023','02/2023','03/2023','04/2023','05/2023','06/2023',
         '07/2023','08/2023','09/2023','10/2023','11/2023','12/2023']

for col in meses:
    df[col] = (df[col].astype(str)
                       .str.replace('.', '', regex=False)
                       .str.replace(',', '.', regex=False)
                       .replace('nan', np.nan)
                       .astype(float))
df.head(20)

In [ ]:
#transforma as 12 colunas de mês em 2 colunas: mes_ano e preco_medio(daquele mês)

df_novo = df.melt(
    id_vars=['produto_nome', 'unidade', 'Nível de Comercialização', 'U.F.'],
    value_vars=meses,
    var_name='mes_ano',
    value_name='preco_medio'
)
df_novo.head(20)


In [ ]:
regiao_map = {
    'AC':'Norte','AP':'Norte','AM':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul'
}
df_regiao = pd.DataFrame(list(regiao_map.items()), columns=['U.F.', 'regiao'])

linhas_antes = len(df_novo)
df_novo = df_novo.merge(df_regiao, on='U.F.', how='left')
print(linhas_antes, len(df_novo))  # confere se bateu — sem duplicar linha

In [ ]:
df_novo.head(20)

In [ ]:
!git init


In [ ]:
!git status

In [ ]:
!git add .

In [ ]:
!git status

In [ ]:
!git config --global user.email "jaycenaiara@gmail.com"
!git config --global user.name "Jayce"

In [ ]:
!git commit -m "comlementei o tratamento de dados para conseguirmos responder as perguntas de análise"

In [ ]:
!git config --global user.name "jaynga8"
!git config --global user.email "jaycenaiara@gmail.com"



In [ ]:
!git checkout -b minhas-modificacoes


In [ ]:
!git branch

In [ ]:
!git add .


In [ ]:
!git status

In [ ]:
!git commit -m "comlementei o tratamento de dados para conseguirmos responder as perguntas de análise"


In [ ]:
!git push --set-upstream origin minhas-modificacaoes

In [ ]:
!git remote add origin https://github.com/harieldalbo/projeto002


In [ ]:
!git push origin minhas-modificacoes

In [ ]:
!git add .



In [ ]:
!git commit -m "comlementei o tratamento de dados para conseguirmos responder as perguntas de análise"


In [ ]:
!git push origin minhas-modificacoes


In [ ]:
!git remote add origin https://github.com/harieldalbo/projeto002

In [ ]:
!git status

In [ ]:
!git branch

In [ ]:
## Análise 1: Qual produto apresentou o maior aumento percentual entre janeiro e dezembro de 2023?

In [ ]:
# Filtrar apenas janeiro e dezembro
df_jan = df_novo[df_novo['mes_ano'] == '01/2023']
df_dez = df_novo[df_novo['mes_ano'] == '12/2023']

# Agrupar por produto e unidade para ter o preço médio nacional em cada mês
preco_jan = df_jan.groupby(['produto_nome', 'unidade'])['preco_medio'].mean().reset_index(name='preco_jan')
preco_dez = df_dez.groupby(['produto_nome', 'unidade'])['preco_medio'].mean().reset_index(name='preco_dez')

# Juntar as duas tabelas
variacao = pd.merge(preco_jan, preco_dez, on=['produto_nome', 'unidade'])

# Calcular a variação percentual
variacao['variacao_pct'] = (variacao['preco_dez'] - variacao['preco_jan']) / variacao['preco_jan'] * 100

# Produto com maior aumento
top_produto = variacao.loc[variacao['variacao_pct'].idxmax()]
print(f"Produto com maior aumento: {top_produto['produto_nome']} ({top_produto['unidade']})")
print(f"Aumento de {top_produto['variacao_pct']:.2f}% (de R${top_produto['preco_jan']:.2f} para R${top_produto['preco_dez']:.2f})")

# Visualizar os 10 maiores aumentos
import matplotlib.pyplot as plt
top10 = variacao.nlargest(10, 'variacao_pct')
plt.figure(figsize=(10,6))
plt.barh(top10['produto_nome'] + ' (' + top10['unidade'] + ')', top10['variacao_pct'])
plt.xlabel('Variação percentual (%)')
plt.title('Top 10 produtos com maior aumento de preço (jan → dez 2023)')
plt.tight_layout()
plt.show()

In [ ]:
## Análise 2: Qual região teve o preço médio mais alto para "ABACAXI PÉROLA (kg)" em 2023?

In [ ]:
# Filtrar o produto específico
abacaxi = df_novo[(df_novo['produto_nome'] == 'ABACAXI PÉROLA') & (df_novo['unidade'] == 'kg')]

# Calcular a média anual por região
media_regiao = abacaxi.groupby('regiao')['preco_medio'].mean().reset_index()
media_regiao = media_regiao.sort_values('preco_medio', ascending=False)

# Exibir
print("Preço médio anual da ABACAXI PÉROLA (kg) por região:")
print(media_regiao)

# Região com maior preço
maior = media_regiao.iloc[0]
print(f"\nRegião com maior preço médio: {maior['regiao']} (R${maior['preco_medio']:.2f})")

# Gráfico de barras
plt.figure(figsize=(8,5))
plt.bar(media_regiao['regiao'], media_regiao['preco_medio'], color='orange')
plt.ylabel('Preço médio (R$)')
plt.title('Preço médio da ABACAXI PÉROLA (kg) por região em 2023')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
## Análise 3: Existe sazonalidade no preço do ABACATE (kg) ao longo dos meses?

In [ ]:
# Filtrar abacate em kg
abacate = df_novo[(df_novo['produto_nome'] == 'ABACATE') & (df_novo['unidade'] == 'kg')]

# Calcular a média nacional por mês (considerando todas as UFs e níveis)
media_mensal = abacate.groupby('mes_ano')['preco_medio'].mean().reset_index()

# Ordenar cronologicamente (já que mes_ano está como string, converter para data)
media_mensal['mes_num'] = pd.to_datetime(media_mensal['mes_ano'], format='%m/%Y').dt.month
media_mensal = media_mensal.sort_values('mes_num')

# Gráfico de linha
plt.figure(figsize=(10,5))
plt.plot(media_mensal['mes_ano'], media_mensal['preco_medio'], marker='o', linestyle='-')
plt.ylabel('Preço médio nacional (R$)')
plt.xlabel('Mês')
plt.title('Variação mensal do preço do ABACATE (kg) em 2023')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Estatísticas
print("Preço médio por mês:")
print(media_mensal)
print(f"\nMês com maior preço: {media_mensal.loc[media_mensal['preco_medio'].idxmax(), 'mes_ano']} (R${media_mensal['preco_medio'].max():.2f})")
print(f"Mês com menor preço: {media_mensal.loc[media_mensal['preco_medio'].idxmin(), 'mes_ano']} (R${media_mensal['preco_medio'].min():.2f})")

In [ ]:
## Análise 4: O preço do arroz no atacado é diferente do varejo?

In [ ]:
# Filtrar produtos que contenham "ARROZ" (exemplo: ARROZ LONGO FINO...) e unidade 'kg'
arroz = df_novo[(df_novo['produto_nome'].str.contains('ARROZ', case=False)) & (df_novo['unidade'] == 'kg')]

# Calcular média por nível de comercialização
media_nivel = arroz.groupby('Nível de Comercialização')['preco_medio'].mean().reset_index()

print("Preço médio do arroz (kg) por nível de comercialização:")
print(media_nivel)

# Gráfico
plt.figure(figsize=(8,5))
plt.bar(media_nivel['Nível de Comercialização'], media_nivel['preco_medio'], color=['green', 'blue', 'red'])
plt.ylabel('Preço médio (R$)')
plt.xlabel('Nível de Comercialização')
plt.title('Comparação de preços do arroz (kg) por canal')
for i, row in media_nivel.iterrows():
    plt.text(row['Nível de Comercialização'], row['preco_medio']+0.5, f'R${row["preco_medio"]:.2f}', ha='center')
plt.tight_layout()
plt.show()

# Comparação direta (atacado vs varejo)
if 'ATACADO' in media_nivel['Nível de Comercialização'].values and 'VAREJO' in media_nivel['Nível de Comercialização'].values:
    preco_atacado = media_nivel[media_nivel['Nível de Comercialização'] == 'ATACADO']['preco_medio'].values[0]
    preco_varejo = media_nivel[media_nivel['Nível de Comercialização'] == 'VAREJO']['preco_medio'].values[0]
    diferenca_pct = (preco_varejo - preco_atacado) / preco_atacado * 100
    print(f"\nO varejo é {diferenca_pct:.2f}% mais caro que o atacado.")
    

In [ ]:
## Análise 5: Quais foram os 5 meses com maior preço médio geral (todos os produtos e UFs)?

In [ ]:
# Média geral simples por mês (sem ponderação por volume de vendas – apenas a média dos preços registrados)
media_geral_mes = df_novo.groupby('mes_ano')['preco_medio'].mean().reset_index()

# Ordenar e pegar os 5 maiores
top5_meses = media_geral_mes.nlargest(5, 'preco_medio')

print("5 meses com maior preço médio geral (todos os produtos):")
print(top5_meses)

# Gráfico de barras
plt.figure(figsize=(8,5))
plt.bar(top5_meses['mes_ano'], top5_meses['preco_medio'], color='purple')
plt.ylabel('Preço médio geral (R$)')
plt.xlabel('Mês')
plt.title('Top 5 meses com maior preço médio em 2023')
for i, row in top5_meses.iterrows():
    plt.text(row['mes_ano'], row['preco_medio']+0.5, f'R${row["preco_medio"]:.2f}', ha='center')
plt.tight_layout()
plt.show()